# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

Reconnect to the persistent vector database

In [4]:
from lib.vector_db import VectorStoreManager
from lib.memory import LongTermMemory

manager = VectorStoreManager(
    openai_api_key=OPENAI_API_KEY,
    persistent_directory="chromadb"
)
vector_store = manager.get_store("udaplay")

assert vector_store is not None, (
    "No 'udaplay' collection found at ./chromadb"
)
long_term_memory = LongTermMemory(manager)
web_knowledge_store = manager.get_or_create_store("web_knowledge")

print(f"Connected. {len(vector_store.get()['ids'])} games available.")
print(f"web_knowledge: {len(web_knowledge_store.get()['ids'])} learned facts")

Connected. 15 games available.
web_knowledge: 0 learned facts


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [5]:
from lib.tooling import tool
import json

@tool
def retrieve_game(query: str) -> str:
    """
    Semantic search: Finds most results in the vector DB.

    Args:
        query: a question about the game industry.

    Returns a JSON string: a list of objects, each with:
        - Platform: e.g. Game Boy, PlayStation 5, Xbox 360
        - Name: name of the game
        - YearOfRelease: year the game was released on that platform
        - Description: additional details about the game
        - similarity: 0-1 similarity score (higher = closer match)
        - source (internal or web_learned)
    """
    n_results = 3
    games = []
    resutls = vector_store.query(query_texts=query, n_results=n_results)
    metas = resutls["metadatas"][0] if resutls["metadatas"] else []
    dists = resutls["distances"][0] if resutls["distances"] else []

    for meta, dist in zip(metas, dists):
        games.append(
            {
                "Platform": meta.get("Platform"),
                "Name": meta.get("Name"),
                "YearOfRelease": meta.get("YearOfRelease"),
                "Description": meta.get("Description"),
                "similarity": round(1 - dist, 3),
                "source": "internal",
            }
        )
    remaining = n_results - len(games)
    if remaining > 0:
        try:
            learned = web_knowledge_store.query(query_texts=[query], n_results=remaining)
            l_docs = learned["documents"][0] if learned["documents"] else []
            l_metas = learned["metadatas"][0] if learned["metadatas"] else []
            l_dists = learned["distances"][0] if learned["distances"] else []
            for content, meta, dist in zip(l_docs, l_metas, l_dists):
                games.append(
                    {
                        "Platform": meta.get("Platform", "unknown"),
                        "Name": meta.get("Name", "unknown"),
                        "YearOfRelease": meta.get("YearOfRelease", "unknown"),
                        "Description": content,
                        "similarity": round(1 - dist, 3),
                        "source": "web_learned",

                    }
                )
        except Exception:
            pass

    return json.dumps(games)

#### Evaluate Retrieval Tool

In [6]:
from pydantic import BaseModel, Field
from lib.llm import LLM
from lib.parsers import PydanticOutputParser

class EvaluationReport(BaseModel):
    is_sufficient: bool = Field(description="Whether the retrieved fames are sufficient to answer the question")
    reason: str = Field(description="Brief explanation of the judgement")

@tool
def evaluate_retrieval(question: str, retrieve_context: str) -> str:
    """
    Assess whether context retrieved from the internal vector DB is
    sufficient to answer the user's question about the video game industry.

    Args:
        question: the original user question.
        retrieved_context: the JSON string returned by retrieve_game.

    Returns a JSON string with:
        - is_sufficient: bool
        - reason: str
    """
    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = f"""You are assessing whether retrieved information is enough to answer a question.

    Question: {question}
    Retrieved context (JSON): {retrieve_context}

    Decide if this retrieved context is sufficient to confidently and accurately
    answer the question. Mark it as NOT sufficiecnt if the context is empty,
    clearly irrelevant, or missing the specific facts the question asks for
    (e.g. wrong game, wrong platform, wrong year, or no results at all).
"""
    response = judge.invoke(
        input=prompt,
        response_format=EvaluationReport
    )
    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)

    return evaluation.model_dump_json()

#### Game Web Search Tool

In [7]:
from tavily import TavilyClient
from lib.documents import Document
import time

_tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def game_web_search(query: str) -> str:
    """
    Search the web for video game industry information not found in the
    internal database - e.g. games not in the local collection, or recent
    industry news.

    Args:
        query: the search query.

    Returns a JSON string: a list of {title, url, content} results.
    """
    response = _tavily_client.search(query=query, max_results=3)
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]
    for r in results:
        if not r.get("content"):
            continue
        try:
            web_knowledge_store.add(Document(
                content=r["content"],
                metadata={
                    "title": r.get("title") or "",
                    "url": r.get("url") or "",
                    "learned_From_query": query,
                    "timestamp": int(time.time()),
                },
            ))
        except Exception:
            pass
    return json.dumps(results)

In [8]:
class SentimentReport(BaseModel):
    game: str = Field(description="The game being assessed")
    sentiment: str = Field(description="Overall sentinment: positive, negative, or mixed")
    summary: str = Field(description="One or two sentence summary of why")


@tool
def analyze_game_sentiment(game_name: str) -> str:
    """
    Estimate the general sentiment/reception of a game, based on a web
    search of reviews and discussion (not a dedicated review dataset -
    this is an approximation, not a rigorous sentiment analysis pipeline).

    Args:
        game_name: the game to assess.

    Returns a JSON string with: game, sentiment ("positive"/"negative"/"mixed"), summary.
    """
    search = _tavily_client.search(query=f"{game_name} game reviews reception opinions", max_results=5)
    snippets = "\n\n".join(
        r.get("content", "") for r in search.get("results", []) if r.get("content")
    )

    judge = LLM(
        model="gpt-4o-mini",
        temperature=0.0
    )
    prompt=f"""Based on these review/discussion snippets about "{game_name}", estimate the
    overall sentiment. This is an approximation from web snippets, not a full
    review corpus - be appropriately hedged in the summary.

    Snippets:
    {snippets if snippets else "(no snippets found)"}
    """ 
    response = judge.invoke(input=prompt, response_format=SentimentReport)
    parser = PydanticOutputParser(model_class=SentimentReport)
    report = parser.parse(response)

    return report.model_dump_json()

In [9]:
class TrendingGames(BaseModel):
    games: str = Field(description="Names of games currently trending, per the search results")
    note: str = Field(description="Brief caveat about how this was determined")

from typing import List

@tool
def detect_trending_games() -> str:
    """
    Get a rough list of currently trending video games, based on a web
    search (not a real trending/sales-data feed - an approximation).

    Returns a JSON string with: games (list of names), note (caveat).
    """
    search = _tavily_client.search(query="trending video games right now", max_results=5)
    snippets = "\n\n".join(
        r.get("content", "") for r in search.get("results", []) if r.get("content")
    )

    judge = LLM(model="gpt-4o-mini", temperature=0.0)
    prompt = f"""Based on these web search snippets, list the video games that appear to be
    currently trending or getting a lot of attention. Note in your response that
    this is derived from a general web search, not a dedicated trending-games feed.

    Snippets:
    {snippets if snippets else "(no snippets found)"}
    """
    response = judge.invoke(input=prompt, response_format=TrendingGames)
    parser = PydanticOutputParser(model_class=TrendingGames)
    result = parser.parse(response)

    return result.model_dump_json()

### Agent

In [10]:
from lib.stateful_agent import StatefulAgent

instructions = """You are UdaPlay, a research assistant for the video game industry.

For every question about a specific game, platform, or release:
1. Call retrieve_game first to search the internal database.
2. Call evaluate_retrieval with the question and the JSON retrieve_game returned,
   to check whether that's actually enough to answer.
3. If evaluate_retrieval says the context is NOT sufficient, call game_web_search
   to find the answer instead.
4. Answer the user's question directly and concisely. State whether your answer
   came from the internal database or previously learned web knowledge or a fresh web search.

Use analyze_game_sentiment when asked how a game was recieved or reviewed.
Use detect_trending_games when asked what's popular or trending right now.
Always caveat sentinment/trending answers as approcimations from web search,
not a dedicated data pipeline.

If you're told something about the user (their preferences, favorite games,
platforms they own), remember it's available to you as "Relevant memory about
this user" on later turns - use it naturally where relevant, don't ignore it.

If the question isn't about games at all, answer normally without using tools."""

agent = StatefulAgent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[
      retrieve_game,
      evaluate_retrieval,
      game_web_search,
      analyze_game_sentiment,
      detect_trending_games],
    long_term_memory=long_term_memory,
    temperature=0.0,
)

Reporting

In [11]:
from lib.state_machine import Run

def print_report(run: Run):
    print(f"=== Run {run.run_id} ===")
    print(f"Started:  {run.start_timestamp}")
    print(f"Finished: {run.end_timestamp}")

    for snap in run.snapshots:
        if snap.step_id in ("__entry__",):
            continue

        if snap.step_id == "memory_recall":
            print(f"=== Agent memory ===")
            print(snap.state_data.get("memory_context") or "none recalled")
            print("----" * 76)

        messages = snap.state_data.get("messages", [])
        if not messages:
            continue
        last = messages[-1]
        role = getattr(last, "role", None)

        if role == "assistant":
            if getattr(last, "tool_calls", None):
                for tc in last.tool_calls:
                    print(f"\n [tool call] {tc.function.name}({tc.function.arguments})")
            if last.content:
                print(f"\n  [assistant] {last.content}")
        elif role == "tool":
            preview = last.content if len(last.content) <= 300 else last.content[:300] + "..."
            print(f" [tool result: {last.name}] {preview}")

    final_state = run.get_final_state()
    final_messages = final_state.get("messages", [])
    final_answer = next(
        (m.content for m in reversed(final_messages)
        if getattr(m, "role", None) == "assistant" and m.content),
        "No answer produced"
    )

    print("\n=== Final Answer ===")
    print(final_answer)

    print("\n=== Structured Answer ===")
    print(final_state.get("structured_answer", "(none)"))
    
    print(f"\nTotal tokens used: {final_state.get('total_tokens', 0)}")

In [12]:
run1 = agent.invoke(
    "What is the most recent Call of Duty game as of 2026?",
    session_id="learn-a",
    owner="idris",
)
print_report(run1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: game_web_search
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 0817a718-69f1-40b3-96e1-1ace498bc407 ===
Started:  2026-09-17 18:30:04.567556
Finished: 2026-09-17 18:30:29.397170
=== Agent memory ===
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving simulation and extensive car selection. If you have any other questions or need more recommendations, feel 

In [13]:
# New session - if the fact was actually learned, this should be answerable
# from retrieve_game alone (source: web_learned), without a second web search.
run2 = agent.invoke(
    "What is the most recent Call of Duty game as of 2026?",
    session_id="learn-b",
    owner="idris",
)
print_report(run2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run e8d30cba-9f62-4d01-9d02-da1b409217a7 ===
Started:  2026-09-17 18:31:26.887005
Finished: 2026-09-17 18:31:31.760959
=== Agent memory ===
- User asked: What is the most recent Call of Duty game as of 2026?
Assistant answered: The most recent Call of Duty game as of 2026 is **Call of Duty: Modern Warfare 4**, which is set to be released on **October 23, 2026**. It will be available on platforms including Nintendo Switch 2, PlayStation 5, Windows, and Xbox Series X/S. This information comes from a recent web search.
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving si

In [ ]:
run2 = agent.invoke(
    "My favorite platform is the PlayStation 1 - keep that in mind for recommendations.",
    session_id="session-a",
    owner="idris",
)
print_report(run1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run eaae35b4-b00e-48a4-971a-70f1136b3241 ===
Started:  2026-09-17 15:58:25.232543
Finished: 2026-09-17 15:58:29.915662
=== Agent memory ===
- User asked: My favorite platform is the PlayStation 1 - keep that in mind for recommendations.
Assistant answered: I've noted that your favorite platform is the PlayStation 1. If you need any recommendations or have questions about games for that platform, just let me know!
- User asked: My favorite platform is the PlayStation 1 - keep that in mind for recommendations.
Assistant answered: I've noted that your favorite platform is the PlayStation 1. If you need any recommendations or have questions about games for that platform, just let me know!
- User asked: My favorite platform is the 

In [14]:
run3 = agent.invoke("How was Cyberpunk 2077 received by players?", session_id="sentiment-demo", owner="idris")
print_report(run3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: analyze_game_sentiment
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 59d2aa63-dacd-4f54-bd80-74846b9f3870 ===
Started:  2026-09-17 18:32:27.882728
Finished: 2026-09-17 18:32:39.519621
=== Agent memory ===
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving simulation and extensive car selection. If you're looking for a great racing experience on your favorite platform, this is an excellent choice! If you have any other questions or need more recommendations, feel free to ask!
- User asked: Recommend a racing game for me.
Assistant answe

In [15]:
run4 = agent.invoke("What games are trending right now?", session_id="trending-demo", owner="idris")
print_report(run4)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: detect_trending_games
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 5d3b3cb7-2822-4f14-8701-0d5443c60237 ===
Started:  2026-09-17 18:33:27.295746
Finished: 2026-09-17 18:33:39.894235
=== Agent memory ===
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving simulation and extensive car selection. If you have any other questions or need more recommendations, feel free to ask!
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaime

Same owner, session_ids (simulating the user coming back later)

In [16]:
run5 = agent.invoke(
    "When Pokémon Gold and Silver was released?",
    session_id="session-a",
    owner="idris",
)
print_report(run5)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run c034fa63-d733-4ab8-b9d3-d46c3c512227 ===
Started:  2026-09-17 18:33:59.931192
Finished: 2026-09-17 18:34:03.958093
=== Agent memory ===
- User asked: When Pokémon Gold and Silver was released?
Assistant answered: Pokémon Gold and Silver was released in 1999 for the Game Boy Color. This information comes from the internal database. If you have any other questions or need recommendations for the PlayStation 1, feel free to ask!
- User asked: When Pokémon Gold and Silver was released?
Assistant answered: Pokémon Gold and Silver was released in 1999 for the Game Boy Color. This information comes from the internal database. If you have any other questions or need recommendations for the PlayStation 1, feel free to ask!
- User a

In [17]:

run6 = agent.invoke(
    "Which one was the first 3D platformer Mario game?",
    session_id="session-b",
    owner="idris",
)
print_report(run6)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 2f7b25f3-4741-4f81-8c06-76f562bf3e94 ===
Started:  2026-09-17 18:34:46.366477
Finished: 2026-09-17 18:34:51.834589
=== Agent memory ===
- User asked: Which one was the first 3D platformer Mario game?
Assistant answered: The first 3D platformer Mario game is "Super Mario 64," which was released in 1996 for the Nintendo 64. This information comes from the internal database. If you have more questions or need recommendations for the PlayStation 1, feel free to ask!
- User asked: Which one was the first 3D platformer Mario game?
Assistant answered: The first 3D platformer Mario game is "Super Mario 64," which was released in 1996 for the Nintendo 64. This information comes from the internal database. If you have more questions

In [18]:
run7 = agent.invoke(
    "Was Mortal Kombat X realeased for Playstation 5?",
    session_id="session-c",
    owner="idris"
)
print_report(run7)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: game_web_search
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 868d279b-95c9-4dc5-9ad3-1caa679769ba ===
Started:  2026-09-17 18:35:13.044709
Finished: 2026-09-17 18:35:33.669026
=== Agent memory ===
- User asked: Was Mortal Kombat X realeased for Playstation 5?
Assistant answered: Mortal Kombat X was not released for PlayStation 5. It was originally released for PlayStation 4, Xbox One, and PC in 2015. My answer is based on a web search, as the internal database did not provide relevant information.
- User asked:

In [19]:
run8 = agent.invoke(
    "Recommend a racing game for me.",
    session_id="session-d",
    owner="idris",
)
print_report(run8)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: memory_recall
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: memory_register
[StateMachine] Terminating: __termination__
=== Run 01e5e2fd-09b2-4805-a7dd-e21947d57061 ===
Started:  2026-09-17 18:36:06.805042
Finished: 2026-09-17 18:36:12.410820
=== Agent memory ===
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving simulation and extensive car selection. If you have any other questions or need more recommendations, feel free to ask!
- User asked: Recommend a racing game for me.
Assistant answered: I recommend "Gran Turismo," which was released in 1997 for the PlayStation 1. It's a highly acclaimed racing game known for its realistic driving simulation and extensive car selection. If you have 

In [27]:
agent.memory.get_all_sessions()

['default', 'session-a', 'session-b', 'session-c', 'session-d']

Visualization

In [20]:
import html as _html

def _svg_bar_chart(counts: dict, title: str, width=600, bar_height=28, gap=10) -> str:
    if not counts:
        return f"<p>No data for {_html.escape(title)}</p>"
    max_count = max(counts.values())
    bars = []
    y = 10
    for label, count in sorted(counts.items(), key=lambda kv: -kv[1]):
        bar_width = int((count / max_count) * (width - 200)) if max_count else 0
        bars.append(f'''
            <text x="0" y="{y + bar_height/2 + 5}" font-size="13">{_html.escape(str(label))}</text>
            <rect x="150" y="{y}" width="{bar_width}" height="{bar_height}" fill="#4C72B0" rx="3"/>
            <text x="{150 + bar_width + 8}" y="{y + bar_height/2 + 5}" font-size="13">{count}</text>
        ''')
        y += bar_height + gap
    return f'<svg width="{width}" height="{y+10}" xmlns="http://www.w3.org/2000/svg">{"".join(bars)}</svg>'


def _run_trace_html(run: Run) -> str:
    rows = []
    for snap in run.snapshots:
        if snap.step_id == "__entry__":
            continue
        messages = snap.state_data.get("messages", [])
        detail = ""
        if messages:
            last = messages[-1]
            role = getattr(last, "role", None)
            if role == "assistant" and getattr(last, "tool_calls", None):
                detail = "; ".join(f"{tc.function.name}({tc.function.arguments})" for tc in last.tool_calls)
            elif role == "assistant" and last.content:
                detail = last.content[:200]
            elif role == "tool":
                detail = f"[{last.name}] {last.content[:200]}"
        rows.append(f'''
            <div class="step">
                <div class="step-id">{_html.escape(snap.step_id)}</div>
                <div class="step-detail">{_html.escape(detail)}</div>
            </div>
            <div class="arrow">&#8595;</div>
        ''')
    return "".join(rows)


def build_dashboard(vector_store, run: Run, output_path="udaplay_dashboard.html"):
    all_games = vector_store.get()
    metas = all_games.get("metadatas", [])
    platform_counts = {}
    for m in metas:
        platform = m.get("Platform", "unknown")
        platform_counts[platform] = platform_counts.get(platform, 0) + 1

    html_doc = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>UdaPlay Dashboard</title>
<style>
  body {{ font-family: -apple-system, sans-serif; max-width: 900px; margin: 40px auto; color: #222; }}
  h1, h2 {{ color: #2c3e50; }}
  .step {{ background: #f5f7fa; border: 1px solid #dde3ea; border-radius: 6px; padding: 10px 14px; }}
  .step-id {{ font-weight: 600; color: #34495e; }}
  .step-detail {{ font-size: 13px; color: #555; margin-top: 4px; white-space: pre-wrap; }}
  .arrow {{ text-align: center; color: #999; margin: 2px 0; }}
</style></head>
<body>
  <h1>UdaPlay Knowledge Base</h1>
  <p>{len(metas)} games across {len(platform_counts)} platforms</p>
  {_svg_bar_chart(platform_counts, "Games per platform")}

  <h1>Run Trace: {_html.escape(run.run_id)}</h1>
  <p>Started {run.start_timestamp} &mdash; Finished {run.end_timestamp}</p>
  {_run_trace_html(run)}
</body></html>"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(html_doc)
    print(f"Dashboard written to {output_path} - open it in a browser.")


build_dashboard(vector_store, run4)

Dashboard written to udaplay_dashboard.html - open it in a browser.
